# Blog figures — Anyscale post (BLOG_ANYSCALE_DRAFT.md)

Infra-story figures. Same conventions as the personal notebook: `$FM_BASE`,
loud skips, PNG+SVG into the repo's `figures/` dir (commit them). Data figures load the same canonical
artifacts so the two posts can't drift apart.

| cell | asset | source |
|---|---|---|
| arch | **A1 / B2-annotated** architecture hero (stage -> scale knob + hardware) | drawn here |
| results | **A2** fulltest results strip (the one benchmark figure this post needs) | fulltest CI jsons |
| knobs | **A3** laptop -> cluster is a config change | `configs/*.yaml`, parsed live |
| distxgb | **A4** distributed-XGBoost story (3 tries + success vs single-node band) | `downstream/xl_fulltest_distxgb*/` |
| stages | **A5** GPUs per pipeline stage (0 -> 8 autoscale story) | `job_full.yaml` facts |
| velocity | **A6** commit-velocity histogram | `git log` of this checkout (labeled) |
| costs | **A7 / B15** cost per run | `RUN_COSTS` — paste from console |

Note (audit): the draft's `distributed_xgb.py` code block is from the research
branch — either check the script in or reframe "every code block is copied
from the template" before publish.


In [ ]:
import os, json, glob, subprocess, sys

for _pkg in ("plotly", "kaleido"):
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd

BASE = os.environ.get("FM_BASE", "/mnt/user_storage/transaction-fm-v2")
_repo_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
FIG_OUT = os.environ.get("FM_FIG_OUT", os.path.join(_repo_root, "figures"))
os.makedirs(FIG_OUT, exist_ok=True)

# Anyscale brand (Notion Brand Hub go/brand): Anyscale Blue is the single accent;
# neutrals do the rest. Muted gray for context series, warm red only for failure.
ACCENT, MUTED, FAIL = "#0066FF", "#a8a6a0", "#c05a4e"
INK, INK2, GRID, SURFACE = "#000000", "#52514e", "#e6e0da", "#ffffff"
CTX_RAMP = {512: "#7fb0ff", 1024: "#0066FF", 2048: "#003c99"}  # ordinal blue ramp

pio.templates["anyscale"] = go.layout.Template(layout=dict(
    font=dict(family="Archivo, Inter, Helvetica, Arial", size=13, color=INK2),
    title=dict(font=dict(size=17, color=INK), x=0.02, xanchor="left"),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    colorway=[ACCENT, MUTED, FAIL, "#4b34b3", "#00a2e9"],
    xaxis=dict(gridcolor=GRID, zeroline=False),
    yaxis=dict(gridcolor=GRID, zeroline=False),
    margin=dict(l=70, r=40, t=80, b=70),
    legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right"),
))
pio.templates.default = "anyscale"

def psave(fig, name, w=1250, h=540):
    fig.write_image(f"{FIG_OUT}/{name}.png", width=w, height=h, scale=2)
    fig.write_image(f"{FIG_OUT}/{name}.svg", width=w, height=h)
    print(f"[saved] {FIG_OUT}/{name}.png + .svg")

def load_results(path):
    if not os.path.exists(path):
        print(f"[SKIP] missing {path}")
        return None
    return json.load(open(path))["results"]

def add_ci_dots(fig, rows, col=1):
    """Dot + 95%-CI whiskers for (label, result-dict, is_ours) rows: whiskers wear the
    point color, accent points get their value annotated above the whisker top."""
    rows = [(l, r, o) for l, r, o in rows if r is not None]
    kw = {"row": 1, "col": col} if getattr(fig, "_grid_ref", None) else {}
    fig.update_xaxes(categoryorder="array", categoryarray=[l for l, _, _ in rows],
                     tickangle=22, **kw)
    for ours in (False, True):
        sub = [(l, r) for l, r, o in rows if o == ours]
        if not sub:
            continue
        c = ACCENT if ours else MUTED
        fig.add_trace(go.Scatter(
            x=[l for l, _ in sub], y=[r["ap"] for _, r in sub], mode="markers",
            marker=dict(size=10, color=c),
            error_y=dict(type="data", color=c, thickness=1.6, width=5,
                         array=[r["ap_ci95"][1] - r["ap"] for _, r in sub],
                         arrayminus=[r["ap"] - r["ap_ci95"][0] for _, r in sub]),
            cliponaxis=False, showlegend=False,
            hovertemplate="%{x}: AP %{y:.4f}<extra></extra>"), **kw)
        if ours:
            for l, r in sub:
                fig.add_annotation(x=l, y=r["ap_ci95"][1], text=f'{r["ap"]:.3f}',
                                   yanchor="bottom", yshift=3, showarrow=False,
                                   font=dict(size=12, color=INK), **kw)

ci100k = {sc: load_results(f"{BASE}/downstream/{sc}/bootstrap_ci.json") for sc, _ in [("full",512),("xl",1024),("xxl",2048)]}
# Table-1 2048 row = the 20-epoch eval (same training budget as 512/1024);
# downstream/xxl holds the 40-ep continuation rerun (table 2's model).
ci100k["xxl"] = (load_results(f"{BASE}/downstream/xxl_old_1783532341/bootstrap_ci.json")
                 or ci100k["xxl"])
cifull = {sc: load_results(f"{BASE}/downstream/{sc}_fulltest/bootstrap_ci.json") for sc, _ in [("full",512),("xl",1024),("xxl",2048)]}
SCALES = [("full", 512), ("xl", 1024), ("xxl", 2048)]
NV_BASELINE, NV_FUSION = 0.1238, 0.1755


## A1 — architecture hero (stage \u2192 scale knob + hardware)\n\nThe anchor figure of the post.

In [ ]:
import shutil, subprocess
d2src = os.path.join(_repo_root, "figures", "src", "architecture.d2")
if shutil.which("d2") and os.path.exists(d2src):
    subprocess.run(["d2", "--layout", "elk", "--pad", "24", d2src, f"{FIG_OUT}/b2_architecture_anyscale.svg"], check=True)
    if shutil.which("rsvg-convert"):
        subprocess.run(["rsvg-convert", "-w", "2400", f"{FIG_OUT}/b2_architecture_anyscale.svg",
                        "-o", f"{FIG_OUT}/b2_architecture_anyscale.png"], check=True)
    print(f"[saved] {FIG_OUT}/b2_architecture_anyscale.svg (+png if rsvg-convert present)")
else:
    print("[SKIP] d2 render — committed figures/b2_architecture_anyscale.* are canonical")


## A2 — the one results figure (fulltest, CI-separated)

In [ ]:
rows = []
for sc, ctx in SCALES:
    r = cifull.get(sc)
    if r:
        rows.append((f"embed→XGB · {ctx}" + (" (40ep)" if ctx == 2048 else ""),
                     r.get("embed_xgb"), True))
base_r = cifull["full"] and cifull["full"].get("baseline")
if rows and base_r:
    fig = go.Figure()
    fig.add_hrect(y0=base_r["ap_ci95"][0], y1=base_r["ap_ci95"][1],
                  fillcolor=GRID, opacity=0.5, line_width=0)
    fig.add_hline(y=base_r["ap"], line=dict(color=INK2, width=1))
    fig.add_annotation(text=f'raw-13 baseline {base_r["ap"]:.3f}', xref="x domain", x=0.02,
                       y=base_r["ap"], xanchor="left", yanchor="bottom", showarrow=False,
                       font=dict(size=12, color=INK2))
    add_ci_dots(fig, rows)
    fig.update_layout(title="The embedding beats the identical-rows raw baseline, CI-separated",
                      yaxis_title="Average precision (full 2.44M-row test)", height=500,
                      margin=dict(b=110, r=70))
    fig.add_annotation(text="Dots = point AP, whiskers = bootstrap 95% CI, band = baseline 95% CI. Identical test rows.",
                       xref="paper", yref="paper", x=0.5, y=-0.22, showarrow=False,
                       font=dict(size=11.5, color=INK2))
    psave(fig, "a2_results", w=950, h=500)
    fig.show()
else:
    print("[SKIP] A2 — need fulltest bootstrap_ci.json for all scales")


## A3 — laptop → cluster is a config change

House rule: numbers in tables ship as **HTML/markdown, never images**. This cell parses
`configs/*.yaml` live and emits the markdown table to paste into the draft (plus an HTML
snippet at `figures/a3_scale_knobs.html`). The old PNG rendering is retired.


In [ ]:
import yaml

KNOB_ROWS = [
    ("cards sampled",        lambda c: f"{c['data']['num_cards']:,}"),
    ("context (txns/window)", lambda c: f"{c['tokenize']['seq_len']:,}"),
    ("d_model / layers",     lambda c: f"{c['model']['d_model']} / {c['model']['n_layers']}"),
    ("pretrain epochs",      lambda c: str(c['pretrain']['epochs'])),
    ("train workers × GPU",  lambda c: f"{c['pretrain']['num_workers']} × " + ("GPU" if c['pretrain']['use_gpu'] else "CPU")),
    ("embed workers (job_full overrides to 8)", lambda c: f"{c['embed']['num_workers']}" + (" (GPU)" if c['embed']['use_gpu'] else " (CPU)")),
]
SHOW = ["smoke", "small", "full", "xl", "xxl"]
cfgs = {s: yaml.safe_load(open(os.path.join(_repo_root, "configs", f"{s}.yaml")))
        for s in SHOW if os.path.exists(os.path.join(_repo_root, "configs", f"{s}.yaml"))}
if cfgs:
    cols = [s for s in SHOW if s in cfgs]
    header = "| knob | " + " | ".join(f"**{c}**" + (" (headline)" if c == "full" else "") for c in cols) + " |"
    md = [header, "|" + "---|" * (len(cols) + 1)]
    for label, fn in KNOB_ROWS:
        md.append(f"| {label} | " + " | ".join(fn(cfgs[c]) for c in cols) + " |")
    table_md = "\n".join(md)
    print(table_md)
    html = "<table>" + "".join(
        "<tr>" + "".join(f"<t{'h' if i == 0 else 'd'}>{cell}</t{'h' if i == 0 else 'd'}>"
                         for cell in row.strip("|").split("|")) + "</tr>"
        for i, row in enumerate(md) if not set(row) <= {"|", "-"}) + "</table>"
    with open(f"{FIG_OUT}/a3_scale_knobs.html", "w") as f:
        f.write(html)
    print(f"\n[saved] {FIG_OUT}/a3_scale_knobs.html — paste the markdown above into the draft")
else:
    print("[SKIP] A3 — configs/ not found")


## A4 — distributed XGBoost: the fusion-at-scale story\n\nAll four ledgered runs from `downstream/xl_fulltest_distxgb*/`; single-node CI band as the bar to clear.

In [ ]:
DIST_RUNS = [
    ("xl_fulltest_distxgb_old_1783719521", "4 workers · hist<br>(first try)"),
    ("xl_fulltest_distxgb_ctrl1w",         "1 worker control · hist"),
    ("xl_fulltest_distxgb",                "4 workers · hist"),
    ("xl_fulltest_distxgb_gpu",            "4 workers · GPU"),
]
runs = []
for d, lab in DIST_RUNS:
    p = f"{BASE}/downstream/{d}/distributed_xgb_metrics.json"
    if os.path.exists(p):
        m = json.load(open(p))
        runs.append((lab, m["ap"]))
    else:
        print(f"[skip run] {p}")
ref = cifull.get("xl") and cifull["xl"].get("embed_xgb")
if runs and ref:
    fig = go.Figure()
    fig.add_hrect(y0=ref["ap_ci95"][0], y1=ref["ap_ci95"][1], fillcolor=GRID,
                  opacity=0.55, line_width=0)
    fig.add_hline(y=ref["ap"], line=dict(color=INK2, width=1))
    fig.add_annotation(text=f'single-node embed→XGB {ref["ap"]:.3f} (95% CI band)',
                       xref="x domain", x=0.02, y=ref["ap_ci95"][1], xanchor="left",
                       yanchor="bottom", showarrow=False, font=dict(size=11.5, color=INK2))
    fig.add_trace(go.Scatter(
        x=[r[0] for r in runs], y=[r[1] for r in runs], mode="markers+text",
        text=[f"{r[1]:.3f}" for r in runs], textposition="top center",
        textfont=dict(size=13, color=INK), texttemplate="%{text}",
        marker=dict(size=12, color=[ACCENT if "GPU" in r[0] else MUTED for r in runs]),
        cliponaxis=False, showlegend=False,
        hovertemplate="%{x}: AP %{y:.4f}<extra></extra>"))
    fig.update_layout(title="Distributed XGBoost on the 1024 embedding: three tries fall short —<br>"
                            "the GPU run lands inside the single-node CI",
                      yaxis_title="Average precision (full 2.44M-row test)",
                      height=520, margin=dict(t=100, r=70))
    psave(fig, "a4_distributed_xgb", w=1000, h=520)
    fig.show()
else:
    print("[SKIP] A4 — need xl_fulltest_distxgb*/distributed_xgb_metrics.json + xl fulltest CI")


## A5 — GPUs follow the pipeline (the 0\u21928 autoscale story)

In [ ]:
# Declarative from job_full.yaml: GPU group (g5.xlarge, min 0 max 8) and CPU worker
# group (m5.4xlarge, min 0 max 4, CPU:0 fence keeps CPU stages off GPU nodes).
STAGES = ["data + vocab", "tokenize", "pretrain", "embed", "XGBoost + reco", "serve"]
GPUS   = [0, 0, 4, 8, 0, 1]
CPU_W  = [4, 4, 0, 4, 0, 0]  # Ray Data CPU tasks: scans, map_groups, embed read side
fig = go.Figure()
for name, ys, color, dashed in (("A10G GPUs held", GPUS, ACCENT, False),
                                ("CPU worker nodes (m5.4xlarge)", CPU_W, INK2, True)):
    fig.add_trace(go.Scatter(
        x=STAGES, y=ys, mode="lines+markers+text", line_shape="hvh",
        line=dict(color=color, width=2.4, dash="dot" if dashed else "solid"),
        marker=dict(size=8, color=color),
        text=[str(v) for v in ys], textposition="top center",
        textfont=dict(size=13, color=color), cliponaxis=False, name=name,
        hovertemplate="%{x}: %{y}<extra>" + name + "</extra>"))
fig.update_layout(title="The cluster reshapes itself per stage — CPU fleet for data, GPU fleet for<br>"
                        "training and inference; min_nodes: 0 scales both to zero",
                  yaxis=dict(title="nodes held", tickvals=[0, 4, 8], range=[-0.6, 9.8]),
                  height=480, margin=dict(t=110))
psave(fig, "a5_stage_gpus", w=1050, h=480)
fig.show()


## A6 — commit velocity, ledgered\n\nThe draft's 161/115/52/45 stats came from memory and don't match this branch — this cell prints git's actual numbers; update the draft to whatever it says.

In [ ]:
import subprocess
try:
    out = subprocess.run(["git", "log", "--pretty=%ad", "--date=short", "--", "."],
                         capture_output=True, text=True, cwd=_repo_root)
    dates = pd.Series(out.stdout.split())
    branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"],
                            capture_output=True, text=True, cwd=_repo_root).stdout.strip()
except Exception:
    dates = pd.Series(dtype=str); branch = "?"
if len(dates):
    counts = dates.value_counts().sort_index()
    fig = go.Figure(go.Bar(
        x=list(counts.index), y=list(counts.values), marker_color=ACCENT,
        text=[str(v) for v in counts.values], textposition="outside",
        hovertemplate="%{x}: %{y} commits<extra></extra>"))
    fig.update_layout(title=f"Campaign velocity — commits/day on `{branch}` (ledgered from git, not memory)",
                      yaxis_title="commits touching this template", height=460,
                      xaxis=dict(type="category", tickangle=30))
    psave(fig, "a6_commit_velocity", w=1050, h=460)
    fig.show()
    print(f"total commits: {counts.sum()}  |  use THESE numbers in the velocity section")
else:
    print("[SKIP] A6 — no git history available here")


## B15 — cost per run

Job costs are not queryable from inside the workspace — pull them from the
Anyscale console (job page \u2192 cost) into `RUN_COSTS` below, then run the cell.
Job IDs from BLOG_ASSETS.md. Skips loudly while any cost is None; once filled
it exports the cost figure and a `run_costs.json` ledger the drafts can cite
(replacing the "$15\u201325 per headline run" placeholder).


In [ ]:
RUN_COSTS = [
    ("512 full run (e2e)",        "<console: fintech-transaction-fm-full>",  None),
    ("1024 run (e2e)",            "prodjob_1nxzlze72xgv7imvgvlw6hk9eb",      None),
    ("2048 run (e2e)",            "prodjob_szkm6hlic3hkhgrn4admydltua",      None),
    ("2048 20→40ep continuation", "prodjob_mr5udb5cnax3qybbr6idup8w2b",      None),
    ("pinned+CUDA eval",          "prodjob_n8tv97crr4nasnsjmw1ne3x2yg",      None),
    ("fulltest evals (3×)",       "<console: job_fulltest*>",                None),
]
missing = [l for l, _, c in RUN_COSTS if c is None]
if missing:
    print(f"[SKIP] B15 — fill costs from the console for: {missing}")
else:
    with open(f"{FIG_OUT}/run_costs.json", "w") as f:
        json.dump([{"run": l, "job_id": j, "usd": c} for l, j, c in RUN_COSTS], f, indent=2)
    labels = [l for l, _, _ in RUN_COSTS][::-1]
    costs = [c for _, _, c in RUN_COSTS][::-1]
    fig = go.Figure(go.Bar(
        x=costs, y=labels, orientation="h", marker_color=ACCENT,
        text=[f"${c:,.0f}" for c in costs], textposition="outside", cliponaxis=False))
    fig.update_layout(title=f"What the campaign actually cost — total ${sum(costs):,.0f}",
                      xaxis_title="cost (USD, Anyscale console)", height=440,
                      margin=dict(l=220))
    psave(fig, "b15_costs", w=1000, h=440)
    fig.show()


---\n**Produced here:** `b2_architecture_anyscale`, `a2_results`, `a3_scale_knobs`, `a4_distributed_xgb`, `a5_stage_gpus`, `a6_commit_velocity`, `b15_costs` (once `RUN_COSTS` is filled).\n\n**Manual:** hero-graphic design pass (Cowork/Figma, using A1 as the content spec), console costs.